Filtrar por region Puno

In [15]:
import pandas as pd

In [ ]:
# df_raw = pd.read_csv("/workspace/notebooks/01_data_cleansing/dataset_filtrado.csv", low_memory=False)
df_raw = pd.read_csv("/workspace/data/collection_peru_csv/data_merge/COLLECT_COMP_PERU.csv", low_memory=False)

In [11]:
# Ver qué variables de región existen
cols_region = [c for c in df_raw.columns if any(x in c.upper()
               for x in ["SUBNATIO", "STRATUM", "REGION", "PUNO"])]
print("Columnas de región encontradas:", cols_region)

# Ver valores únicos de SUBNATIO si existe
if "SUBNATIO" in df_raw.columns:
    print("\nValores únicos de SUBNATIO:")
    print(df_raw["SUBNATIO"].value_counts().head(20))

if "STRATUM" in df_raw.columns:
    print("\nValores únicos de STRATUM:")
    print(df_raw["STRATUM"].value_counts().head(20))

Columnas de región encontradas: ['STRATUM', 'SUBNATIO', 'REGION']

Valores únicos de SUBNATIO:
SUBNATIO
6040000    6968
Name: count, dtype: int64

Valores únicos de STRATUM:
STRATUM
PER01    3929
PER03    1805
PER02    1190
PER04      44
Name: count, dtype: int64


Tratando de identificar las regiones en el dataset

In [12]:
df_raw = pd.read_csv("/workspace/data/collection_peru_csv/data_merge/COLLECT_COMP_PERU.csv", low_memory=False)

print(f"Total de filas: {len(df_raw):,}")
print(f"\nValores únicos de CNT: {df_raw['CNT'].unique()}")
print(f"\nDistribución de STRATUM:")
print(df_raw["STRATUM"].value_counts())
print(f"\nDistribución de SUBNATIO:")
print(df_raw["SUBNATIO"].value_counts())

Total de filas: 6,968

Valores únicos de CNT: <ArrowStringArray>
['PER']
Length: 1, dtype: str

Distribución de STRATUM:
STRATUM
PER01    3929
PER03    1805
PER02    1190
PER04      44
Name: count, dtype: int64

Distribución de SUBNATIO:
SUBNATIO
6040000    6968
Name: count, dtype: int64


In [16]:
df_raw = pd.read_csv("/workspace/data/collection_peru_csv/data_merge/COLLECT_COMP_PERU.csv", low_memory=False)

# Buscar columnas relacionadas con ubicación, comunidad, región, área
keywords = ["URBAN", "RURAL", "CITY", "TOWN", "VILLAGE", "COMMUN",
            "LOCATION", "AREA", "REGION", "GEOG", "SC001", "SC013"]

cols_geo = [c for c in df_raw.columns
            if any(k in c.upper() for k in keywords)]

print("Columnas potencialmente geográficas:")
for col in cols_geo:
    n_unicos = df_raw[col].nunique()
    print(f"  {col:<25} — {n_unicos} valores únicos")
    print(f"    {df_raw[col].value_counts().head(5).to_dict()}")
    print()

Columnas potencialmente geográficas:
  REGION                    — 1 valores únicos
    {'Peru': 6968}

  SC001Q01TA                — 6 valores únicos
    {'A small town (3 000 to about 15 000 people)': 1886, 'A village, hamlet or rural area (fewer than 3 000 people)': 1834, 'A town (15 000 to about 100 000 people)': 1688, 'A city (100 000 to about 1 000 000 people)': 1174, 'A megacity (with over 10 000 000 people)': 221}

  SC013Q01TA                — 2 valores únicos
    {'A public school(This is a school managed directly or indirectly by a public education authority, government agency, or g': 5254, 'A private school(This is a school managed directly or indirectly by a non-government organisation; e.g. a church, trade': 1714}



In [17]:
import pandas as pd

df_raw = pd.read_csv("/workspace/data/collection_peru_csv/data_merge/COLLECT_COMP_PERU.csv", low_memory=False)
df_lca = pd.read_csv("/workspace/notebooks/01_data_cleansing/dataset_lca_clases.csv", low_memory=False)

# Agregar variables geográficas al dataset de clases
df_lca["tipo_comunidad"] = df_raw["SC001Q01TA"].values
df_lca["tipo_escuela"]   = df_raw["SC013Q01TA"].values

# Simplificar etiquetas largas
mapa_comunidad = {
    "A village, hamlet or rural area (fewer than 3 000 people)"        : "Rural",
    "A small town (3 000 to about 15 000 people)"                      : "Pueblo pequeño",
    "A town (15 000 to about 100 000 people)"                          : "Ciudad pequeña",
    "A city (100 000 to about 1 000 000 people)"                       : "Ciudad",
    "A large city (1 000 000 to about 10 000 000 people)"              : "Ciudad grande",
    "A megacity (with over 10 000 000 people)"                         : "Megaciudad",
}
mapa_escuela = {c: ("Pública" if "public" in c.lower() else "Privada")
                for c in df_lca["tipo_escuela"].unique()}

df_lca["tipo_comunidad"] = df_lca["tipo_comunidad"].map(mapa_comunidad)
df_lca["tipo_escuela"]   = df_lca["tipo_escuela"].map(mapa_escuela)

# Tabla 1 — tipo de comunidad por clase latente
tabla_comunidad = pd.crosstab(
    df_lca["clase_lca"],
    df_lca["tipo_comunidad"],
    normalize="index"
).round(3) * 100

print("Distribución de tipo de comunidad por clase latente (%):")
print(tabla_comunidad.to_string())

# Tabla 2 — tipo de escuela por clase latente
tabla_escuela = pd.crosstab(
    df_lca["clase_lca"],
    df_lca["tipo_escuela"],
    normalize="index"
).round(3) * 100

print("\nDistribución de tipo de escuela por clase latente (%):")
print(tabla_escuela.to_string())

# Tabla 3 — rendimiento promedio por tipo de comunidad
print("\nRendimiento promedio por tipo de comunidad:")
print(df_lca.groupby("tipo_comunidad")["score_rendimiento"]
      .agg(["mean","std","count"]).round(1).to_string())

Distribución de tipo de comunidad por clase latente (%):
tipo_comunidad  Ciudad  Ciudad grande  Ciudad pequeña  Megaciudad  Pueblo pequeño  Rural
clase_lca                                                                               
0                 14.1            2.0            27.2         3.4            30.9   22.4
1                 31.2            2.6            25.5         1.4            21.3   18.0
2                 11.3            3.8            24.6         2.6            21.4   36.3
3                  6.8            0.9            18.6         3.7            27.0   43.1
4                 32.4            2.6            21.8         3.1            23.3   16.7
5                 13.0            3.0            26.8         3.7            31.5   21.9

Distribución de tipo de escuela por clase latente (%):
tipo_escuela  Privada  Pública
clase_lca                     
0                 1.8     98.2
1                91.4      8.6
2                 0.3     99.7
3                 0.

In [13]:
# df_raw = pd.read_csv("/workspace/data/collection_csv/datos_formateados_STU_TIM.csv", low_memory=False)

# Carga el dataset de estudiantes ORIGINAL de PISA antes de cualquier filtro
df_original = pd.read_csv("/workspace/data/collection_csv/datos_formateados_STU_TIM.csv", low_memory=False)

# Ver todos los valores de STRATUM para Perú
peru = df_original[df_original["CNT"] == "PER"]
print(f"Total estudiantes Perú: {len(peru):,}")
print(f"\nValores únicos de STRATUM en Perú:")
print(peru["STRATUM"].value_counts().to_string())
print(f"\nValores únicos de SUBNATIO en Perú:")
print(peru["SUBNATIO"].value_counts().to_string())
print(f"\nValores únicos de REGION en Perú (si existe):")
if "REGION" in peru.columns:
    print(peru["REGION"].value_counts().to_string())

Total estudiantes Perú: 6,968

Valores únicos de STRATUM en Perú:
STRATUM
PER01    3929
PER03    1805
PER02    1190
PER04      44

Valores únicos de SUBNATIO en Perú:
SUBNATIO
6040000    6968

Valores únicos de REGION en Perú (si existe):
REGION
Peru    6968


Pruebas con el dataset MINEDU

In [ ]:
peru = pd.read_spss("/workspace/data/minedu_pisa/SAV/BD_DE_ESTUDIANTES.sav")

# Ver todos los valores de STRATUM para Perú
# peru = df_original[df_original["CNT"] == "PER"]
print(f"Total estudiantes Perú: {len(peru):,}")
print(f"\nValores únicos de STRATUM en Perú:")
print(peru["STRATUM"].value_counts().to_string())
print(f"\nValores únicos de SUBNATIO en Perú:")
print(peru["SUBNATIO"].value_counts().to_string())
print(f"\nValores únicos de REGION en Perú (si existe):")
print(peru["SUBNATIO"].value_counts().to_string())
print(f"\nValores únicos de REGION en Perú (si existe):")
if "REGION" in peru.columns:
    print(peru["REGION"].value_counts().to_string())

Total estudiantes Perú: 6,968

Valores únicos de STRATUM en Perú:
STRATUM
PER - stratum 01: Public,Urban     3929
PER - stratum 03: Private,Urban    1805
PER - stratum 02: Public,Rural     1190
PER - stratum 04: Private,Rural      44

Valores únicos de SUBNATIO en Perú:
SUBNATIO
Peru    6968

Valores únicos de REGION en Perú (si existe):
REGION
Peru    6968


In [ ]:
peru = pd.read_spss("/workspace/data/minedu_pisa/SAV/1. BD DE ESCUELA.sav")

print(f"Total estudiantes Perú: {len(peru):,}")
print(f"\nValores únicos de STRATUM en Perú:")
print(peru["STRATUM"].value_counts().to_string())
print(f"\nValores únicos de SUBNATIO en Perú:")
print(peru["SUBNATIO"].value_counts().to_string())
print(f"\nValores únicos de REGION en Perú (si existe):")
print(peru["SUBNATIO"].value_counts().to_string())
print(f"\nValores únicos de REGION en Perú (si existe):")
if "REGION" in peru.columns:
    print(peru["REGION"].value_counts().to_string())

Total estudiantes Perú: 336

Valores únicos de STRATUM en Perú:
STRATUM
PER - stratum 01: Public,Urban     146
PER - stratum 03: Private,Urban     95
PER - stratum 02: Public,Rural      91
PER - stratum 04: Private,Rural      4

Valores únicos de SUBNATIO en Perú:
SUBNATIO
Peru    336

Valores únicos de REGION en Perú (si existe):
SUBNATIO
Peru    336

Valores únicos de REGION en Perú (si existe):
REGION
Peru    336


In [19]:
peru = pd.read_spss("/workspace/data/minedu_pisa/SAV/1. BD DE ESCUELA.sav")

print("Todas las columnas del dataset de escuelas:")
for col in peru.columns:
    n_unicos = peru[col].nunique()
    muestra = peru[col].value_counts().head(3).to_dict()
    print(f"  {col:<30} — {n_unicos} únicos — {muestra}")

Todas las columnas del dataset de escuelas:
  CNT                            — 1 únicos — {'Peru': 336}
  CNTRYID                        — 1 únicos — {'Peru': 336}
  CNTSCHID                       — 336 únicos — {60400001.0: 1, 60400002.0: 1, 60400003.0: 1}
  CYC                            — 1 únicos — {'08MS': 336}
  NatCen                         — 1 únicos — {'Peru': 336}
  STRATUM                        — 4 únicos — {'PER - stratum 01: Public,Urban': 146, 'PER - stratum 03: Private,Urban': 95, 'PER - stratum 02: Public,Rural': 91}
  SUBNATIO                       — 1 únicos — {'Peru': 336}
  REGION                         — 1 únicos — {'Peru': 336}
  OECD                           — 1 únicos — {'No': 336}
  ADMINMODE                      — 1 únicos — {'Computer': 336}
  LANGTEST_QQQ                   — 1 únicos — {'Spanish': 336}
  SC001Q01TA                     — 6 únicos — {'A village, hamlet or rural area (fewer than 3 000 people)': 123, 'A small town (3 000 to about 15 000 peop